In [ ]:
%reload_ext autoreload
%autoreload 2

import os
import numpy as np
import dill as pickle

import jax
jax.config.update("jax_enable_x64", True)
import jax.numpy as jnp
from jax.example_libraries import stax

from numpyro.infer import SVI, Trace_ELBO, autoguide
from numpyro import optim
import optax

from fpp.models.np_model import NPModel
from fpp.utils.posterior import multi_corner, dnds_posterior

import matplotlib as mpl
import matplotlib.pyplot as plt
mpl.rc_file("./matplotlibrc")

# SVI Process: Evolving Posterior

Visualize how the SVI guide/posterior evolves during training by loading
checkpointed params saved by `fit_fermi_svi_process.py`.

In [ ]:
# --- Paths ---
svi_process_dir = "../outputs/svi_process"
plots_dir = "../outputs/svi_process/plots"
os.makedirs(plots_dir, exist_ok=True)

# --- Load checkpoint metadata ---
save_steps = np.load(f"{svi_process_dir}/svi_save_steps.npy")
losses = np.load(f"{svi_process_dir}/svi_losses.npy")
print(f"Checkpoint steps: {save_steps}")
print(f"Total training steps: {len(losses)}")

In [ ]:
# --- Initialize model + guide (must match fit_fermi_svi_process.py) ---
data = jnp.array(
    np.load("../data/fermi_data_573w/fermi_data_128/fermidata_counts.npy"),
    dtype=jnp.int32,
)

m = NPModel(
    data=data,
    psf_tag='king',
    n_exp=7,
    diffuse_names=["ModelO", "ModelA", "ModelF"],
)

guide = autoguide.AutoIAFNormal(
    m.model,
    num_flows=5,
    hidden_dims=[128, 128],
    nonlinearity=stax.Tanh,
)
m.guide = guide

# Initialize SVI so the guide is properly set up
optimizer = optim.optax_to_numpyro(optax.chain(optax.clip(1.), optax.adam(3e-4)))
loss_fn = Trace_ELBO(num_particles=16, vectorize_particles=True)
svi = SVI(m.model, guide, optimizer, loss_fn)
_ = svi.init(jax.random.PRNGKey(42), data=data)

In [ ]:
# --- Sample from guide at each checkpoint ---
num_samples = 50000
rng_key = jax.random.PRNGKey(0)

samples_by_step = {}
for step in save_steps:
    step = int(step)
    params = pickle.load(open(f"{svi_process_dir}/svi_params_step{step}.p", "rb"))
    raw_samples = guide.sample_posterior(
        rng_key=rng_key,
        params=params,
        sample_shape=(num_samples,),
    )
    samples_by_step[step] = m.expand_samples(raw_samples)
    print(f"Step {step}: sampled {num_samples} draws")

## Loss curve

In [ ]:
fig, ax = plt.subplots(figsize=(8, 3))
ax.plot(losses, lw=0.5)
for step in save_steps:
    if step > 0:
        ax.axvline(step, color='r', ls='--', alpha=0.4, lw=0.8)
ax.set(xlabel='SVI step', ylabel='ELBO loss', title='Training loss with checkpoint markers')
fig.tight_layout()
fig.savefig(f"{plots_dir}/svi_loss_curve.png", dpi=200)

## Evolving corner plots

Overlay posteriors from selected checkpoints to see how the guide sharpens.

In [ ]:
label_latex_d = {
    'S_pib': r'$S_\mathrm{pib}$',
    'S_ics': r'$S_\mathrm{ics}$',
    'S_iso': r'$S_\mathrm{iso}$',
    'S_bub': r'$S_\mathrm{bub}$',
    'S_gce': r'$S_\mathrm{gce}$',
    'S_nfw': r'$S_\mathrm{nfw}$',
    'Sps_dsk': r'$S_\mathrm{dsk}^\mathrm{ps}$',
    'Sps_gce': r'$S_\mathrm{gce}^\mathrm{ps}$',
    'f_bulge_poiss': r'$f_\mathrm{bulge}^\mathrm{pois}$',
    'f_bulge_ps':    r'$f_\mathrm{bulge}^\mathrm{ps}$',
    'gamma_poiss': r'$\gamma^\mathrm{pois}$',
    'gamma_ps':    r'$\gamma^\mathrm{ps}$',
    'C': r'$C$',
    'zs': r'$z_s$',
}

labels = [
    'S_pib', 'S_ics', 'S_gce', 'Sps_dsk', 'Sps_gce',
    'f_bulge_poiss', 'f_bulge_ps', 'gamma_poiss', 'gamma_ps', 'C', 'zs',
]

In [ ]:
# Select a subset of checkpoints for the overlay corner plot
corner_steps = [s for s in save_steps if s in [0, 100, 500, 2000, 5000, 10000]]
if not corner_steps:
    corner_steps = save_steps  # fallback: use all

cmap = mpl.colormaps['viridis']
n_show = len(corner_steps)

s_in = {}
colors_dict = {}
legend_dict = {}
for idx, step in enumerate(corner_steps):
    step = int(step)
    key = f'step_{step}'
    s_in[key] = {k: np.array(samples_by_step[step][k]) for k in labels}
    colors_dict[key] = cmap(idx / max(n_show - 1, 1))
    legend_dict[key] = f'Step {step}'

multi_corner(
    s_in, labels,
    colors_dict=colors_dict,
    legend_dict=legend_dict,
    legend_loc=(0.15, 0.94),
    labels=[label_latex_d[k] for k in labels],
    label_kwargs={'fontsize': 30},
    save_fn=f'{plots_dir}/svi_process_corner.png',
)

## Evolving dN/dS

In [ ]:
labels_func = lambda k: [f'Sps_{k}', f'n1_{k}', f'n2_{k}', f'n3_{k}', f'sb1_{k}', f'lambdas_{k}']

# Select steps for dN/dS evolution
dnds_steps = [s for s in save_steps if s in [500, 1000, 2000, 5000, 10000]]
if not dnds_steps:
    dnds_steps = [s for s in save_steps if s > 0][-4:]  # last 4 non-zero

fig, axs = plt.subplots(1, 2, figsize=(10, 4), sharey=True)

cmap = mpl.colormaps['viridis']
n_show = len(dnds_steps)

for idx, step in enumerate(dnds_steps):
    step = int(step)
    color = cmap(idx / max(n_show - 1, 1))
    samples = samples_by_step[step]

    for ik, k in enumerate(['gce', 'dsk']):
        try:
            s, dnds_med, dnds_68, dnds_95 = dnds_posterior(
                samples=samples, theta_keys=labels_func(k), plot=False,
            )
            axs[ik].plot(s, dnds_med, color=color, label=f'Step {step}')
            axs[ik].fill_between(s, dnds_68[0], dnds_68[1], alpha=0.15, fc=color, ec='none')
        except Exception as e:
            print(f"Step {step}, {k}: skipped ({e})")

for ik, (ax, title) in enumerate(zip(axs, ['GCE point source', 'Disk point source'])):
    ax.set(xlim=(1, 50), ylim=(1e-7, 1), xscale='log', yscale='log')
    ax.set(xlabel='$S$ [counts]', title=title)
    if ik == 0:
        ax.set(ylabel='$dN/dS$ [counts$^{-1}$]')
    ax.legend(frameon=False, fontsize=8)

fig.tight_layout()
fig.savefig(f'{plots_dir}/svi_process_dnds.png', dpi=300)

## Marginal evolution for selected parameters

In [ ]:
sel_params = ['S_pib', 'S_ics', 'Sps_gce', 'Sps_dsk']
n_params = len(sel_params)

fig, axs = plt.subplots(1, n_params, figsize=(3.5 * n_params, 3))

cmap = mpl.colormaps['viridis']
n_show = len(save_steps)

for idx, step in enumerate(save_steps):
    step = int(step)
    color = cmap(idx / max(n_show - 1, 1))
    alpha = 0.3 + 0.7 * idx / max(n_show - 1, 1)
    for ip, pname in enumerate(sel_params):
        vals = np.array(samples_by_step[step][pname])
        axs[ip].hist(
            vals, bins=60, density=True, histtype='step',
            color=color, alpha=alpha, lw=1.0,
            label=f'Step {step}' if ip == 0 else None,
        )
        axs[ip].set(xlabel=label_latex_d[pname], yticks=[])

axs[0].legend(frameon=False, fontsize=7, loc='upper left')
fig.suptitle('Marginal posterior evolution', y=1.02)
fig.tight_layout()
fig.savefig(f'{plots_dir}/svi_process_marginals.png', dpi=200, bbox_inches='tight')